<a href="https://colab.research.google.com/github/Arunaeshkanna/My-Portfolio/blob/main/newsemproject17.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-generativeai python-dotenv pypdf scikit-learn joblib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.6/329.6 kB 5.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
PERSIST_DIR = "/content/drive/MyDrive/orion_ai_store"
import os
os.makedirs(PERSIST_DIR, exist_ok=True)
print("Persistent directory ready:", PERSIST_DIR)


Persistent directory ready: /content/drive/MyDrive/orion_ai_store


In [ ]:
import os

# 🔑 PUT YOUR GEMINI API KEY HERE
os.environ["GEMINI_API_KEY"] = "your api"


In [ ]:
import google.generativeai as genai

genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

llm = genai.GenerativeModel("gemini-2.5-flash")

print(llm.generate_content("Say hello").text)


Hello!


In [ ]:
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

def ingestion_agent(uploaded_files):
    documents = []

    for filename in uploaded_files:
        if filename.endswith(".pdf"):
            reader = PdfReader(filename)
            for page in reader.pages:
                text = page.extract_text()
                if text:
                    documents.append(text)

        elif filename.endswith(".txt"):
            with open(filename, "r", encoding="utf-8") as f:
                documents.append(f.read())

    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=5000
    )
    vectors = vectorizer.fit_transform(documents)

    joblib.dump(
        (vectorizer, vectors, documents),
        f"{PERSIST_DIR}/tfidf_store.pkl"
    )

    return len(documents)


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def retrieval_agent(query, k=3):
    vectorizer, vectors, docs = joblib.load(
        f"{PERSIST_DIR}/tfidf_store.pkl"
    )

    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, vectors)[0]

    top_idx = np.argsort(scores)[-k:][::-1]
    context = "\n".join(docs[i] for i in top_idx)

    return context


In [ ]:
def decision_agent(context):
    # Decide if enough information exists
    return len(context.strip()) > 200


In [ ]:
def answering_agent(context, question):
    prompt = f"""
Answer ONLY using the context below.

Context:
{context}

Question:
{question}

If the answer is not in the context, say "I don't know".
"""
    response = llm.generate_content(prompt)
    return response.text


In [ ]:
from google.colab import files

uploaded = files.upload()


Saving python notes.pdf to python notes.pdf


In [ ]:
count = ingestion_agent(uploaded.keys())
print(f"✅ Ingested {count} document sections")


✅ Ingested 154 document sections


In [ ]:
while True:
    question = input("Ask ORION-AI (type exit to stop): ")

    if question.lower() == "exit":
        break

    context = retrieval_agent(question)

    if not decision_agent(context):
        print("\n🤖 ORION-AI:")
        print("I don't know based on the available documents.")
    else:
        answer = answering_agent(context, question)
        print("\n🤖 ORION-AI:")
        print(answer)

    print("-" * 60)


Ask ORION-AI (type exit to stop): what is list in python

🤖 ORION-AI:
I don't know.
------------------------------------------------------------
Ask ORION-AI (type exit to stop): what is an integer

🤖 ORION-AI:
The integer numbers (e.g. 2, 4, 20) have type int. They are contrasted with numbers that have a fractional part.
------------------------------------------------------------
Ask ORION-AI (type exit to stop): what is lists

🤖 ORION-AI:
Lists are a mutable type, meaning it is possible to change their content. They are a sequence data type that supports operations like indexing, slicing, and concatenation. Elements can be added to the end of a list using the `append()` method, and assignment to slices can change the size of the list or clear it entirely. Lists can also contain other lists (nesting). Their elements are usually homogeneous and are accessed by iterating over the list. While they can be used as a queue, they are not efficient for operations involving inserts or pops fr

**final new code**

In [ ]:
!pip install google-generativeai pypdf scikit-learn joblib gradio streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 65.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import google.generativeai as genai

PERSIST_DIR = "/content/drive/MyDrive/orion_ai_store"
os.makedirs(PERSIST_DIR, exist_ok=True)

os.environ["GEMINI_API_KEY"] = "your api"
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

llm = genai.GenerativeModel("gemini-2.5-flash")


In [ ]:
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib
import numpy as np


In [ ]:
def ingestion_agent(files):
    documents = []

    for file in files:
        if file.name.endswith(".pdf"):
            reader = PdfReader(file)
            for page in reader.pages:
                text = page.extract_text()
                if text:
                    documents.append(text)

        elif file.name.endswith(".txt"):
            documents.append(file.read().decode("utf-8"))

    vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)
    vectors = vectorizer.fit_transform(documents)

    joblib.dump(
        (vectorizer, vectors, documents),
        f"{PERSIST_DIR}/tfidf_store.pkl"
    )

    return f"Ingested {len(documents)} chunks"


In [ ]:
def retrieval_agent(query, k=3):
    vectorizer, vectors, docs = joblib.load(
        f"{PERSIST_DIR}/tfidf_store.pkl"
    )

    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, vectors)[0]
    idx = np.argsort(scores)[-k:][::-1]
    return "\n".join(docs[i] for i in idx)


In [ ]:
def decision_agent(context):
    return len(context.strip()) > 200


In [ ]:
def answering_agent(question):
    context = retrieval_agent(question)

    if not decision_agent(context):
        return "I don't know based on the documents."

    prompt = f"""
Answer ONLY using the context.

Context:
{context}

Question:
{question}
"""
    return llm.generate_content(prompt).text


In [ ]:
import gradio as gr

with gr.Blocks(title="ORION-AI (Agentic RAG)") as demo:

    gr.Markdown("## 🤖 ORION-AI – Agentic RAG System (Gemini 2.5 Flash)")

    with gr.Tab("📥 Ingest Documents"):
        file_input = gr.File(
            file_types=[".pdf", ".txt"],
            file_count="multiple"
        )
        ingest_btn = gr.Button("Ingest")
        ingest_output = gr.Textbox()

        ingest_btn.click(
            fn=ingestion_agent,
            inputs=file_input,
            outputs=ingest_output
        )

    with gr.Tab("🔍 Ask Questions"):
        question = gr.Textbox(label="Ask a question")
        answer = gr.Textbox(label="Answer")

        question.submit(
            fn=answering_agent,
            inputs=question,
            outputs=answer
        )

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a162723dd9e651abc6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**final**
